In [9]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup

In [10]:
# Load the data 

# Column names for binary classification PCL data
column_names = ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']

# Reading the data, skipping the first 4 lines of disclaimer
# delimiter is set to \t for tab-separated values
df_pcl = pd.read_csv('data/dontpatronizeme_pcl.tsv', 
                 sep='\t', 
                 skiprows=4, 
                 names=column_names, 
                 index_col=False,
                 quoting=3)


# data load for multi label classification

span_columns = [
    'par_id', 'art_id', 'text', 'keyword', 'country_code', 
    'span_start', 'span_finish', 'span_text', 'pcl_category', 'num_annotators'
]

# Load the data
df_categories= pd.read_csv('data/dontpatronizeme_categories.tsv', 
                       sep='\t', 
                       skiprows=4, 
                       names=span_columns, 
                       index_col=False,
                       quoting=3) # quoting=3 tells pandas to ignore quotes to avoid splitting text mid-sentence

# Training labels
df_train_labels = pd.read_csv('data/train_semeval_parids-labels.csv', index_col=False)

df_dev_labels = pd.read_csv('data/dev_semeval_parids-labels.csv', index_col=False)
                             


In [11]:
# Binary PCL classifcation
# Create the new binary column 'pcl_presence'
df_pcl['pcl_presence'] = df_pcl['label'].apply(lambda x: 0 if x in [0, 1] else 1)

import ast
df_train_labels['label'] = df_train_labels['label'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_dev_labels['label'] = df_dev_labels['label'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [12]:
df_train_labels.head()

,par_id,label
0,4341,"[1, 0, 0, 1, 0, 0, 0]"
1,4136,"[0, 1, 0, 0, 0, 0, 0]"
2,10352,"[1, 0, 0, 0, 0, 1, 0]"
3,8279,"[0, 0, 0, 1, 0, 0, 0]"
4,1164,"[1, 0, 0, 1, 1, 1, 0]"


In [13]:
# merge train and dev labels with pcl presence info
import pandas as pd
import ast


def prepare_final_dataset(df_labels, df_main_text):
    # 1. Merge to get the text, keyword, and country code based on par_id
    df_merged = pd.merge(df_labels, df_main_text[['par_id', 'text', 'keyword', 'country_code',"pcl_presence"]], 
                         on='par_id', how='left')
    
    # Format the input text (Keyword + Country + Text)
    # Using RoBERTa/DeBERTa's separator token </s> 
    df_merged['model_input'] = (
        df_merged['keyword'].astype(str) + " </s> " + 
        df_merged['country_code'].astype(str) + " </s> " + 
        df_merged['text'].astype(str)
    )
    
    # Keep only the columns we actually need for the model
    return df_merged[['par_id', 'model_input', 'pcl_presence', 'label']]

# Apply the function to both your train and dev sets!
print("Preparing Train Set...")
train_data = prepare_final_dataset(df_train_labels, df_pcl)

print("Preparing Dev Set...")
dev_data = prepare_final_dataset(df_dev_labels, df_pcl)

# Let's verify the output
print("\n--- FINAL TRAIN DATA FORMAT ---")
print(train_data.head(3))

Preparing Train Set...
Preparing Dev Set...

--- FINAL TRAIN DATA FORMAT ---
   par_id                                        model_input  pcl_presence  \
0    4341  poor-families </s> gb </s> The scheme saw an e...             1   
1    4136  homeless </s> za </s> Durban 's homeless commu...             1   
2   10352  poor-families </s> lk </s> The next immediate ...             1   

                   label  
0  [1, 0, 0, 1, 0, 0, 0]  
1  [0, 1, 0, 0, 0, 0, 0]  
2  [1, 0, 0, 0, 0, 1, 0]  


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import ast

class PCLMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df['model_input'].tolist()
        self.binary_labels = df['pcl_presence'].tolist()
        self.multi_labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        # Extract Binary Label (0 or 1)
        # We use float32 because Binary Cross Entropy expects floats
        binary_label = torch.tensor([self.binary_labels[idx]], dtype=torch.float32)

        # Extract Multi-Label (The list of 7 numbers)

        m_label = self.multi_labels[idx]
        if isinstance(m_label, str):
            m_label = ast.literal_eval(m_label)
        multi_label_tensor = torch.tensor(m_label, dtype=torch.float32)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'binary_labels': binary_label,
            'multi_labels': multi_label_tensor
        }

# Initialize Tokenizer and Loaders
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
train_dataset = PCLMultiTaskDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super(BinaryFocalLoss, self).__init__()
        self.alpha = alpha  # Weight for the positive class (PCL)
        self.gamma = gamma  # Focusing parameter for hard examples

    def forward(self, logits, targets):
        # Calculate standard Binary Cross Entropy Loss
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        # Calculate probabilities
        pt = torch.exp(-bce_loss) 
        
        # Apply Focal Loss formula
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super(SupervisedContrastiveLoss, self).__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        # Normalize embeddings (puts them on a uniform sphere)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        # Calculate similarity matrix
        similarity_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature

        # clamping values
        similarity_matrix = torch.clamp(similarity_matrix, min=-50.0, max=50.0)
        
        # Find which sentences share the same binary label
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(embeddings.device)
        mask.fill_diagonal_(0) # Remove self-matching
        
        # Compute Contrastive Loss
        exp_sim = torch.exp(similarity_matrix) * (1 - torch.eye(labels.shape[0]).to(embeddings.device))
        log_prob = similarity_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-5)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-5)
        
        return -mean_log_prob_pos.mean()

In [8]:
from transformers import AutoModel

class MultiTaskDeBERTaPCL(nn.Module):
    def __init__(self, model_name='microsoft/deberta-v3-base', num_categories=7):
        super(MultiTaskDeBERTaPCL, self).__init__()
        
        # The Base Transformer
        self.deberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.deberta.config.hidden_size
        self.dropout = nn.Dropout(0.3)
        
        # Head 1: Binary Output (1 node)
        self.binary_head = nn.Linear(hidden_size, 1)
        
        # Head 2: Multi-Label Output (7 nodes)
        self.category_head = nn.Linear(hidden_size, num_categories)

    def forward(self, input_ids, attention_mask):
        # 1. Get contextual embeddings from DeBERTa
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # 2. Extract the [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)

        # 3. Pass through both classification heads
        binary_logits = self.binary_head(pooled_output)
        category_logits = self.category_head(pooled_output)
        
        return binary_logits, category_logits, pooled_output

In [3]:
# Check available CUDA devices and memory
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    for i in range(num_devices):
        device = torch.cuda.device(i)
        total_mem = torch.cuda.get_device_properties(i).total_memory / 1024**3  # Convert to GB
        allocated_mem = torch.cuda.memory_allocated(i) / 1024**3  # Convert to GB
        free_mem = total_mem - allocated_mem
        
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"Total Memory: {total_mem:.1f}GB")
        print(f"Allocated Memory: {allocated_mem:.1f}GB")
        print(f"Free Memory: {free_mem:.1f}GB")
        
        if free_mem < 8:
            print(f"Warning: GPU {i} has less than 8GB of free VRAM!")
        else:
            print(f"Using GPU {i} with {free_mem:.1f}GB free VRAM")
            break 
    device = torch.device(f"cuda:{i}")
else:
    print("Warning: No CUDA devices available - running on CPU only")
    device = torch.device("cpu")


GPU 0: Tesla T4
Total Memory: 14.6GB
Allocated Memory: 0.0GB
Free Memory: 14.6GB
Using GPU 0 with 14.6GB free VRAM


In [10]:
LR = 2e-5

model = MultiTaskDeBERTaPCL().to(device).float()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

epochs = 3

# total_steps = len(train_loader) * EPOCHS
# # Warmup for the first 10% of training steps
# num_warmup_steps = int(0.1 * total_steps)

# scheduler = get_linear_schedule_with_warmup(
#     optimizer, 
#     num_warmup_steps=num_warmup_steps, 
#     num_training_steps=total_steps
# )

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Setup device, model, optimizer, and loss functions
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Initialize our three loss functions
focal_criterion = BinaryFocalLoss()
scl_criterion = SupervisedContrastiveLoss()
# Standard BCE for the 7 categories
multi_label_criterion = nn.BCEWithLogitsLoss() 

from torch.cuda.amp import autocast, GradScaler

# Initialize the Gradient Scaler for Mixed Precision
scaler = GradScaler()


model.train()

for epoch in range(epochs):
    total_loss = 0
    print(f"--- Starting Epoch {epoch + 1}/{epochs} ---")
    
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        binary_labels = batch['binary_labels'].to(device)
        multi_labels = batch['multi_labels'].to(device)
        
        optimizer.zero_grad()
        
        # 1. AUTOCAST: The GPU automatically handles 16-bit/32-bit math safely
        with autocast():
            bin_logits, cat_logits, embeddings = model(input_ids, attention_mask)
            
            # Calculate Losses
            loss_focal = focal_criterion(bin_logits, binary_labels)
            loss_multi = multi_label_criterion(cat_logits, multi_labels)
            
            # SCL Safety Check (needs at least 2 PCL examples to compare)
            if binary_labels.sum() > 1:
                loss_scl = scl_criterion(embeddings, binary_labels.view(-1).long())
            else:
                loss_scl = torch.tensor(0.0).to(device) 
            
            # Combine losses
            joint_loss = (0.8 * loss_focal) + (0.1 * loss_scl) + (0.1 * loss_multi)
        
        # 2. SCALED BACKPROPAGATION: Prevents the NaN explosion!
        scaler.scale(joint_loss).backward()
        
        # Unscale before clipping so the clipping threshold is accurate
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 3. Step optimizer and update scaler
        scaler.step(optimizer)
        scaler.update()
        # scheduler.step()
        
        total_loss += joint_loss.item()
        
        # Print an update every 100 batches so you know it's working!
        if (step + 1) % 100 == 0:
            print(f"Batch {step + 1}/{len(train_loader)} - Loss: {joint_loss.item():.4f}")
            
    print(f"Epoch {epoch + 1} Complete | Avg Joint Loss: {total_loss / len(train_loader):.4f}\n")

torch.save(model.state_dict(), "models/deberta_scl_focalloss.pt")


/tmp/ipykernel_1648/106893901.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1648/106893901.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


--- Starting Epoch 1/3 ---
Batch 100/524 - Loss: 0.0232
Batch 200/524 - Loss: 0.0366
Batch 300/524 - Loss: 0.0261
Batch 400/524 - Loss: 0.3812
Batch 500/524 - Loss: 0.3278
Epoch 1 Complete | Avg Joint Loss: 0.1752

--- Starting Epoch 2/3 ---
Batch 100/524 - Loss: 0.3437
Batch 200/524 - Loss: 0.3528
Batch 300/524 - Loss: 0.0253
Batch 400/524 - Loss: 0.0188
Batch 500/524 - Loss: 0.0238
Epoch 2 Complete | Avg Joint Loss: 0.1641

--- Starting Epoch 3/3 ---
Batch 100/524 - Loss: 0.0068
Batch 200/524 - Loss: 0.2918
Batch 300/524 - Loss: 0.0327
Batch 400/524 - Loss: 0.4083
Batch 500/524 - Loss: 0.2462
Epoch 3 Complete | Avg Joint Loss: 0.1455



In [12]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report

def evaluate_and_tune_threshold(model, dev_loader, device):
    print("Extracting probabilities from Dev Set...")
    model.eval() # Turn off dropout for evaluation
    
    all_probs = []
    all_labels = []
    
    # Turn off gradient tracking to speed up inference
    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            binary_labels = batch['binary_labels'].to(device)
            
            # Forward pass: We only care about the first output (Binary Logits)
            bin_logits, _, _ = model(input_ids, attention_mask)
            
            # Convert raw logits to probabilities (0.0 to 1.0)
            probs = torch.sigmoid(bin_logits)
            
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(binary_labels.cpu().numpy())
            
    y_true = np.array(all_labels).flatten()
    y_probs = np.array(all_probs).flatten()
    
    print("Sweeping thresholds to find peak F1-Score...")
    best_threshold = 0.5
    best_f1 = 0.0
    
    # Test every threshold from 0.10 to 0.90 in steps of 0.01
    thresholds = np.arange(0.1, 0.9, 0.01)
    
    for thresh in thresholds:
        preds = (y_probs >= thresh).astype(int)
        # Calculate F1 for the positive class (1)
        current_f1 = f1_score(y_true, preds, pos_label=1, zero_division=0)
        
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = thresh
            
    print(f"\nBEST THRESHOLD: {best_threshold:.2f}")
    print(f"PEAK F1-SCORE:  {best_f1:.4f}")
    
    # Print the final detailed report using the best threshold
    final_preds = (y_probs >= best_threshold).astype(int)
    print("\n--- FINAL CLASSIFICATION REPORT ---")
    print(classification_report(y_true, final_preds, target_names=['Non-PCL (0)', 'PCL (1)']))
    
    return best_threshold, best_f1

# --- RUN IT! ---
# Make sure you have created your dev_loader first!
dev_dataset = PCLMultiTaskDataset(dev_data, tokenizer)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)

best_thresh, best_f1 = evaluate_and_tune_threshold(model, dev_loader, device)

Extracting probabilities from Dev Set...
Sweeping thresholds to find peak F1-Score...

🔥 BEST THRESHOLD: 0.50
🏆 PEAK F1-SCORE:  0.5101

--- FINAL CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

 Non-PCL (0)       0.96      0.91      0.93      1895
     PCL (1)       0.43      0.63      0.51       199

    accuracy                           0.88      2094
   macro avg       0.69      0.77      0.72      2094
weighted avg       0.91      0.88      0.89      2094

